# Track1039: fresh quality-data retrain

Attach the existing PRIVATE **VGEC26-quality-track1039-private-20261019.zip** dataset. Internet ON, GPU T4 x2. Repository-read GITHUB_TOKEN required; HF_TOKEN optional, WANDB_API_KEY enables numeric/resource monitoring. Save Version / Run All. No toggles or previous notebook output required. Extracted legacy uploads remain supported; no nested archives are needed.

The committed recipe trains **Sailor2-20B classifier → Qwen2.5-14B corrector**, NF4 QLoRA / FP16 compute, five full epochs each. Seed42, unchanged quality data, same78 development rows excluded from fitting,2034 PRIVATE rows for inference only. No baseline comparison, preflight, saved-checkpoint score replay, folds, hints or decoder sweep. All choices live in the committed configs.

Download the single **run/results.zip** from the final link; it uses both freshly trained models. Keep the whole output for adapters, metrics, histories and append-only partial prediction journals. Private classification runs just after classifier training to avoid downloading20B again after14B. Corrector generation uses batches up to4, reducing on inference OOM. Root W&B run stays alive through export and monitors system resources.

No controller wall/stage time limit is imposed. Kaggle's own session limit still applies; the larger microbatches and reduced generation overhead have not been benchmarked. The original track1038 quality experiment and reporting backup remain untouched. Full details: agents/experiments/FAST_RETRAIN_20261020.md.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
SOURCE_ROOT = Path("/kaggle/working")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Fetch the committed source securely into a fresh directory.
from urllib.parse import urlsplit
import tempfile

with tempfile.TemporaryDirectory() as authentication:
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    token = None
    if urlsplit(REPO_URL).hostname == "github.com":
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            raise RuntimeError("Enable GITHUB_TOKEN in Kaggle Secrets with repository read access.") from None
        if not token:
            raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
        askpass = Path(authentication) / "git-askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                           '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="gecc-source-", dir=SOURCE_ROOT))
    result = subprocess.run(["git", "-c", "credential.helper=", "clone", "--depth", "1",
                             "--single-branch", "--branch", BRANCH, "--", REPO_URL, str(PROJECT_DIR)],
                            env=env, capture_output=True, text=True)
    returncode = result.returncode
    detail = result.stderr.replace(token, "[REDACTED]") if token else result.stderr
    del env, token, result
    if returncode:
        raise RuntimeError("Cannot clone source; check Internet, branch, and GITHUB_TOKEN access. " + detail.strip())
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gecc_notebook", PROJECT_DIR / "tools/notebook.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
TRAINING_PYTHON = launcher.setup_kaggle(PROJECT_DIR)


In [ ]:
RUN_DIR = launcher.launch(PROJECT_DIR, TRAINING_PYTHON)
